In [ ]:
"""
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

class Informer(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=64, n_layers=4, n_heads=8):
        super(Informer, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.n_heads = n_heads

        self.input_fc = nn.Linear(input_size, hidden_size)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_size, nhead=n_heads, batch_first=True,dropout=0.3),
            num_layers=n_layers
        )
        self.decoder = nn.Linear(hidden_size, output_size)


    def forward(self, x):
        x = self.input_fc(x)
        x = self.encoder(x)
        x = self.decoder(x[:, -1, :])
        return x

# Veriyi yükleme
df = pd.read_csv('/content/drive/MyDrive/KoreanDataset_5.0.csv')

df['Date'] = pd.to_datetime(df['Date'])
df['Date_ordinal'] = df['Date'].map(lambda x: x.toordinal())

features = ['Date_ordinal', 'AveragePrice', 'WTIPrice', 'BrentPrice', 'tmax', 'tmin', 'tavg', 'wspd', 'pres']
products = ['Latte Varieties', 'Sweets and Cakes', 'Croissant Varieties', 'Fruity and Refreshing Drinks', 'Snacks', 'Classic Breads and Toasts']

from scipy.stats import zscore
df = df[(np.abs(zscore(df[features])) < 3).all(axis=1)]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
#scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(df[features].values)

from scipy.stats import zscore



def create_dataset(X, y, window_size=35, group_size=7):
    X_data, y_data = [], []
    for i in range(len(X) - window_size - group_size + 1):
        X_window = X[i:i + window_size].reshape(-1, group_size, X.shape[1])
        X_avg = np.mean(X_window, axis=1)  # 7 gün ortalaması
        y_avg = np.mean(y[i + window_size:i + window_size + group_size]) # 36-42. günlerin ortalaması
        X_data.append(X_avg)
        y_data.append(y_avg)
    return np.array(X_data), np.array(y_data)

results = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for product in products:
    print(f"Eğitiliyor: {product}")
    y = df[product].values

    X_window, y_window = create_dataset(X_scaled, y)
    X_train, X_test, y_train, y_test = train_test_split(X_window, y_window, test_size=0.2, shuffle=True)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    model = Informer(input_size=X_train.shape[2], output_size=1).to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.0001,weight_decay=1e-4)
    loss_fn = nn.MSELoss()


    epochs = 1000
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            inputs, targets = batch
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                inputs, targets = batch
                outputs = model(inputs).squeeze()
                loss = loss_fn(outputs, targets)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(test_loader):.4f}")

    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs, targets = batch
            outputs = model(inputs).squeeze()
            y_pred.extend(outputs.cpu().numpy())
            y_true.extend(targets.cpu().numpy())

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    results[product] = {"RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}

# Sonuçları DataFrame olarak gösterme
results_df = pd.DataFrame(results).T

print("\nModel Performans Sonuçları:")
print(results_df)

# Sonuçları görselleştirme
fig, axes = plt.subplots(4, 1, figsize=(10, 20))
metrics = ['RMSE', 'MSE', 'MAE', 'R2']

for i, metric in enumerate(metrics):
    axes[i].bar(results_df.index, results_df[metric], width=0.4, color='blue')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_xlabel('Ürünler')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
"""

In [ ]:
"""
from google.colab import drive
drive.mount('/content/drive')
"""

In [ ]:
"""
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

class Informer(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=64, n_layers=4, n_heads=8):
        super(Informer, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.n_heads = n_heads

        self.input_fc = nn.Linear(input_size, hidden_size)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_size, nhead=n_heads, batch_first=True),
            num_layers=n_layers
        )
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.input_fc(x)
        x = self.encoder(x)
        x = self.decoder(x[:, -1, :])
        return x

# Veriyi yükleme
df = pd.read_csv('/content/KoreanDataset_5.0.csv')

df['Date'] = pd.to_datetime(df['Date'])
df['Date_ordinal'] = df['Date'].map(lambda x: x.toordinal())

features = ['Date_ordinal', 'AveragePrice', 'WTIPrice', 'BrentPrice', 'tmax', 'tmin', 'tavg', 'wspd', 'pres']
products = ['Latte Varieties']

scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(df[features].values)

def create_dataset(X, y, window_size=35, group_size=7):
    X_data, y_data = [], []
    for i in range(len(X) - window_size - group_size + 1):
        X_window = X[i:i + window_size].reshape(-1, group_size, X.shape[1])
        X_avg = np.mean(X_window, axis=1)  # 7 gün ortalaması
        y_avg = np.mean(y[i + window_size:i + window_size + group_size]) # 36-42. günlerin ortalaması
        X_data.append(X_avg)
        y_data.append(y_avg)
    return np.array(X_data), np.array(y_data)

results = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for product in products:
    print(f"Eğitiliyor: {product}")
    y = df[product].values

    X_window, y_window = create_dataset(X_scaled, y)
    X_train, X_test, y_train, y_test = train_test_split(X_window, y_window, test_size=0.2, shuffle=False)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    model = Informer(input_size=X_train.shape[2], output_size=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    epochs = 1000
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            inputs, targets = batch
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                inputs, targets = batch
                outputs = model(inputs).squeeze()
                loss = loss_fn(outputs, targets)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(test_loader):.4f}")

    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs, targets = batch
            outputs = model(inputs).squeeze()
            y_pred.extend(outputs.cpu().numpy())
            y_true.extend(targets.cpu().numpy())

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    results[product] = {"RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}

# Sonuçları DataFrame olarak gösterme
results_df = pd.DataFrame(results).T

print("\nModel Performans Sonuçları:")
print(results_df)

# Sonuçları görselleştirme
fig, axes = plt.subplots(4, 1, figsize=(10, 20))
metrics = ['RMSE', 'MSE', 'MAE', 'R2']

for i, metric in enumerate(metrics):
    axes[i].bar(results_df.index, results_df[metric], width=0.4, color='blue')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_xlabel('Ürünler')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
"""

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import zscore

# Informer Model Tanımı
class Informer(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=64, n_layers=4, n_heads=8):
        super(Informer, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.n_heads = n_heads

        self.input_fc = nn.Linear(input_size, hidden_size)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_size, nhead=n_heads, batch_first=True, dropout=0.3),
            num_layers=n_layers
        )
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.input_fc(x)
        x = self.encoder(x)
        x = self.decoder(x[:, -1, :])
        return x

# Veriyi yükleme
df = pd.read_csv('/content/KoreanDataset_5.0.csv')

df['Date'] = pd.to_datetime(df['Date'])
df['Date_ordinal'] = df['Date'].map(lambda x: x.toordinal())

# Kullanılacak özellikler ve ürünler
features = ['Date_ordinal', 'AveragePrice', 'WTIPrice', 'BrentPrice', 'tmax', 'tmin', 'tavg', 'wspd', 'pres']
products = ['Latte Varieties', 'Sweets and Cakes', 'Croissant Varieties']

# Z-score ile uç değerleri temizleme
df = df[(np.abs(zscore(df[features])) < 3).all(axis=1)]

# Veriyi standardize etme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].values)

# Veri setini pencereleme
def create_dataset(X, y, window_size=35, group_size=7):
    X_data, y_data = [], []
    for i in range(len(X) - window_size - group_size + 1):
        X_window = X[i:i + window_size].reshape(-1, group_size, X.shape[1])
        X_avg = np.mean(X_window, axis=1)  # 7 günlük ortalama
        y_avg = np.mean(y[i + window_size:i + window_size + group_size]) # 36-42. günlerin ortalaması
        X_data.append(X_avg)
        y_data.append(y_avg)
    return np.array(X_data), np.array(y_data)

# Modeli eğitme ve test etme
results = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Sonuçları kaydetmek için liste
all_results = {}

for product in products:
    print(f"Eğitiliyor: {product}")

    # Hedef değişkeni seçme
    y = df[product].values

    # Veri setini oluşturma
    X_window, y_window = create_dataset(X_scaled, y)
    X_train, X_test, y_train, y_test = train_test_split(X_window, y_window, test_size=0.2, shuffle=True)

    # Veriyi PyTorch tensörüne çevirme
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    # DataLoader ile veri yükleme
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Modeli oluşturma
    model = Informer(input_size=X_train.shape[2], output_size=1).to(device)

    # Optimizasyon ve kayıp fonksiyonu
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    # Modeli eğitme
    epochs = 1000
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            inputs, targets = batch
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                inputs, targets = batch
                outputs = model(inputs).squeeze()
                loss = loss_fn(outputs, targets)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(test_loader):.4f}")

    # Modeli test etme
    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs, targets = batch
            outputs = model(inputs).squeeze()
            y_pred.extend(outputs.cpu().numpy())
            y_true.extend(targets.cpu().numpy())

    # Performans metriklerini hesaplama
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # Sonuçları saklama
    results[product] = {"RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}
    all_results[product] = {"y_true": y_true, "y_pred": y_pred}

# Sonuçları DataFrame olarak gösterme
results_df = pd.DataFrame(results).T
print("\nModel Performans Sonuçları:")
print(results_df)

# Sonuçları görselleştirme
fig, axes = plt.subplots(4, 1, figsize=(10, 20))
metrics = ['RMSE', 'MSE', 'MAE', 'R2']

for i, metric in enumerate(metrics):
    axes[i].bar(results_df.index, results_df[metric], width=0.4, color='blue')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_xlabel('Ürünler')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

with open("test_vs_estimated_values.txt", "w") as file:
    for product in products:
        file.write(f"Ürün: {product}\n")
        file.write(f"{'Gerçek Değer':<15} {'Tahmin Değeri':<15}\n")
        file.write("-" * 30 + "\n")

        for true_val, pred_val in zip(all_results[product]['y_true'], all_results[product]['y_pred']):
            file.write(f"{true_val:<15.4f} {pred_val:<15.4f}\n")

        file.write("\n\n")

print("\nTest ve tahmin edilen değerler 'test_vs_estimated_values.txt' dosyasına kaydedildi.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import zscore

# Informer Model Tanımı
class Informer(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=64, n_layers=4, n_heads=8):
        super(Informer, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.n_heads = n_heads

        self.input_fc = nn.Linear(input_size, hidden_size)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_size, nhead=n_heads, batch_first=True, dropout=0.3),
            num_layers=n_layers
        )
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.input_fc(x)
        x = self.encoder(x)
        x = self.decoder(x[:, -1, :])
        return x

# Veriyi yükleme
df = pd.read_csv('/content/KoreanDataset_5.0.csv')

df['Date'] = pd.to_datetime(df['Date'])
df['Date_ordinal'] = df['Date'].map(lambda x: x.toordinal())

# Kullanılacak özellikler ve ürünler
features = ['Date_ordinal', 'AveragePrice', 'WTIPrice', 'BrentPrice', 'tmax', 'tmin', 'tavg', 'wspd', 'pres']
products = ['Fruity and Refreshing Drinks', 'Snacks', 'Classic Breads and Toasts']

# Z-score ile uç değerleri temizleme
df = df[(np.abs(zscore(df[features])) < 3).all(axis=1)]

# Veriyi standardize etme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].values)

# Veri setini pencereleme
def create_dataset(X, y, window_size=35, group_size=7):
    X_data, y_data = [], []
    for i in range(len(X) - window_size - group_size + 1):
        X_window = X[i:i + window_size].reshape(-1, group_size, X.shape[1])
        X_avg = np.mean(X_window, axis=1)  # 7 günlük ortalama
        y_avg = np.mean(y[i + window_size:i + window_size + group_size]) # 36-42. günlerin ortalaması
        X_data.append(X_avg)
        y_data.append(y_avg)
    return np.array(X_data), np.array(y_data)

# Modeli eğitme ve test etme
results = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Sonuçları kaydetmek için liste
all_results = {}

for product in products:
    print(f"Eğitiliyor: {product}")

    # Hedef değişkeni seçme
    y = df[product].values

    # Veri setini oluşturma
    X_window, y_window = create_dataset(X_scaled, y)
    X_train, X_test, y_train, y_test = train_test_split(X_window, y_window, test_size=0.2, shuffle=True)

    # Veriyi PyTorch tensörüne çevirme
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    # DataLoader ile veri yükleme
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Modeli oluşturma
    model = Informer(input_size=X_train.shape[2], output_size=1).to(device)

    # Optimizasyon ve kayıp fonksiyonu
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    # Modeli eğitme
    epochs = 1000
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            inputs, targets = batch
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                inputs, targets = batch
                outputs = model(inputs).squeeze()
                loss = loss_fn(outputs, targets)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(test_loader):.4f}")

    # Modeli test etme
    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs, targets = batch
            outputs = model(inputs).squeeze()
            y_pred.extend(outputs.cpu().numpy())
            y_true.extend(targets.cpu().numpy())

    # Performans metriklerini hesaplama
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # Sonuçları saklama
    results[product] = {"RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}
    all_results[product] = {"y_true": y_true, "y_pred": y_pred}

# Sonuçları DataFrame olarak gösterme
results_df = pd.DataFrame(results).T
print("\nModel Performans Sonuçları:")
print(results_df)

# Sonuçları görselleştirme
fig, axes = plt.subplots(4, 1, figsize=(10, 20))
metrics = ['RMSE', 'MSE', 'MAE', 'R2']

for i, metric in enumerate(metrics):
    axes[i].bar(results_df.index, results_df[metric], width=0.4, color='blue')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_xlabel('Ürünler')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Test ve tahmin değerlerini txt dosyasına kaydetme
with open("test_vs_estimated_values_2.txt", "w") as file:
    for product in products:
        file.write(f"Ürün: {product}\n")
        file.write(f"{'Gerçek Değer':<15} {'Tahmin Değeri':<15}\n")
        file.write("-" * 30 + "\n")

        for true_val, pred_val in zip(all_results[product]['y_true'], all_results[product]['y_pred']):
            file.write(f"{true_val:<15.4f} {pred_val:<15.4f}\n")

        file.write("\n\n")

print("\nTest ve tahmin edilen değerler 'test_vs_estimated_values.txt' dosyasına kaydedildi.")
